## Load & Inspect ChEMBL → UniProt → Gene mapping


# Phase 6

In [7]:
import json
import pandas as pd

# load mechanism JSON
with open(
    r"C:\Users\deep8\breast_cancer_project_folder\chembl_mechanism.json",
    "r",
    encoding="utf-8"
) as f:
    mech = json.load(f)

# extract mechanisms list
mech_list = mech["mechanisms"]

len(mech_list)


1000

In [9]:
# convert mechanisms to dataframe
mech_df = pd.DataFrame(mech_list)

mech_df.shape, mech_df.columns


((1000, 17),
 Index(['action_type', 'binding_site_comment', 'direct_interaction',
        'disease_efficacy', 'max_phase', 'mec_id', 'mechanism_comment',
        'mechanism_of_action', 'mechanism_refs', 'molecular_mechanism',
        'molecule_chembl_id', 'parent_molecule_chembl_id', 'record_id',
        'selectivity_comment', 'site_id', 'target_chembl_id',
        'variant_sequence'],
       dtype='object'))

## Parse target.json & map targets → gene symbols


In [12]:
import json
import pandas as pd

# load target metadata
with open(
    r"C:\Users\deep8\breast_cancer_project_folder\chembl_target.json",
    "r",
    encoding="utf-8"
) as f:
    tgt = json.load(f)

targets = tgt["targets"]

len(targets)


1000

In [14]:
# extract target → gene symbol mapping
rows = []
for t in targets:
    tid = t.get("target_chembl_id")
    comps = t.get("target_components", [])
    for c in comps:
        gene = c.get("gene_symbol")
        if gene:
            rows.append({"target_chembl_id": tid, "gene_symbol": gene})

tgt_gene_map = pd.DataFrame(rows).drop_duplicates()

tgt_gene_map.shape, tgt_gene_map.head()


((0, 0),
 Empty DataFrame
 Columns: []
 Index: [])

In [18]:
# inspect one target entry to understand structure
targets[0].keys()


dict_keys(['cross_references', 'organism', 'pref_name', 'species_group_flag', 'target_chembl_id', 'target_components', 'target_type', 'tax_id'])

In [22]:
import json
import pprint

pprint.pprint(targets[0])


{'cross_references': [],
 'organism': 'Homo sapiens',
 'pref_name': 'Maltase-glucoamylase',
 'species_group_flag': False,
 'target_chembl_id': 'CHEMBL2074',
 'target_components': [{'accession': 'O43451',
                        'component_description': 'Maltase-glucoamylase',
                        'component_id': 434,
                        'component_type': 'PROTEIN',
                        'relationship': 'SINGLE PROTEIN',
                        'target_component_synonyms': [{'component_synonym': '3.2.1.20',
                                                       'syn_type': 'EC_NUMBER'},
                                                      {'component_synonym': 'Alpha-1,4-glucosidase',
                                                       'syn_type': 'UNIPROT'},
                                                      {'component_synonym': 'Maltase-glucoamylase',
                                                       'syn_type': 'UNIPROT'},
                                       

In [24]:
pprint.pprint(targets[0]["target_components"])


[{'accession': 'O43451',
  'component_description': 'Maltase-glucoamylase',
  'component_id': 434,
  'component_type': 'PROTEIN',
  'relationship': 'SINGLE PROTEIN',
  'target_component_synonyms': [{'component_synonym': '3.2.1.20',
                                 'syn_type': 'EC_NUMBER'},
                                {'component_synonym': 'Alpha-1,4-glucosidase',
                                 'syn_type': 'UNIPROT'},
                                {'component_synonym': 'Maltase-glucoamylase',
                                 'syn_type': 'UNIPROT'},
                                {'component_synonym': 'MGA',
                                 'syn_type': 'GENE_SYMBOL_OTHER'},
                                {'component_synonym': 'MGAM',
                                 'syn_type': 'GENE_SYMBOL'},
                                {'component_synonym': 'MGAML',
                                 'syn_type': 'GENE_SYMBOL_OTHER'},
                                {'component_synonym': 'Sy

In [26]:
rows = []

for t in targets:
    for comp in t.get("target_components", []):
        gene = None
        
        # extract gene symbol
        for syn in comp.get("target_component_synonyms", []):
            if syn["syn_type"] == "GENE_SYMBOL":
                gene = syn["component_synonym"]
                break
        
        if gene:
            rows.append({
                "target_chembl_id": t["target_chembl_id"],
                "gene_symbol": gene,
                "uniprot": comp.get("accession")
            })

chembl_targets_df = pd.DataFrame(rows).drop_duplicates()
chembl_targets_df.head(), chembl_targets_df.shape


(  target_chembl_id gene_symbol uniprot
 0       CHEMBL2074        MGAM  O43451
 1       CHEMBL1971       ABCC9  O60706
 2       CHEMBL1827       PDE5A  O76074
 3       CHEMBL1859     CACNA1H  O95180
 4        CHEMBL202        DHFR  P00374,
 (979, 3))

# Build **Drug → Target list

In [29]:
drug_targets = (
    chembl_targets_df
    .groupby("target_chembl_id")["gene_symbol"]
    .apply(list)
    .reset_index(name="targets")
)

drug_targets.head(), drug_targets.shape

(  target_chembl_id   targets
 0       CHEMBL1777  [MT-CYB]
 1       CHEMBL1778   [IL2RA]
 2       CHEMBL1780   [ERG11]
 3       CHEMBL1781    [TOP1]
 4       CHEMBL1782    [FDPS],
 (979, 2))

In [35]:


import pandas as pd

# load PPI edge list
ppi = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\PPI_TCGA_BRCA_STRING_HQ.csv"
)

# extract all genes in the PPI
ppi_nodes = set(ppi["gene1"]).union(set(ppi["gene2"]))

len(ppi_nodes)


10095

In [37]:
drug_targets_ppi = drug_targets.copy()

drug_targets_ppi["targets_ppi"] = drug_targets_ppi["targets"].apply(
    lambda genes: [g for g in genes if g in ppi_nodes]
)

# remove drugs with no PPI-supported targets
drug_targets_ppi = drug_targets_ppi[
    drug_targets_ppi["targets_ppi"].str.len() > 0
]

drug_targets_ppi.shape


(522, 3)

In [39]:
drug_targets_ppi.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Drug_Targets_PPI_Filtered.csv",
    index=False
)

# PHASE 6.2 — Network Proximity Features

In [ ]:
and in 6.2 it was 
Target based network feature
for each drug,subtype
-no of target
mean pagerank of target
max pagerank of target
target essentiality depmap


In [108]:
# feature 1: number of PPI-supported targets per drug
drug_targets_ppi["n_targets"] = drug_targets_ppi["targets_ppi"].apply(len)

drug_targets_ppi[["n_targets"]].describe(), drug_targets_ppi.shape

(       n_targets
 count      522.0
 mean         1.0
 std          0.0
 min          1.0
 25%          1.0
 50%          1.0
 75%          1.0
 max          1.0,
 (522, 4))

In [112]:
import pandas as pd

pr_her2_df = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_PPI_Pagerank.csv",
    index_col=0
)

pr_her2_df.shape, pr_her2_df.head()


((9875, 1),
         pagerank
 SRC     0.001975
 TP53    0.001818
 EGFR    0.001521
 RPS27A  0.001392
 EP300   0.001352)

In [114]:
# build gene → pagerank dict for HER2
her2_pr = pr_her2_df["pagerank"].to_dict()

# mean pagerank of targets per drug (HER2)
drug_targets_ppi["HER2_mean_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: sum(her2_pr[g] for g in genes) / len(genes)
)

drug_targets_ppi[["HER2_mean_pr"]].describe()


,HER2_mean_pr
count,522.000000
mean,0.000191
std,0.000205
min,0.000004
25%,0.000078
50%,0.000134
75%,0.000216
max,0.001975


In [116]:
# max pagerank of targets per drug (HER2)
drug_targets_ppi["HER2_max_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: max(her2_pr[g] for g in genes)
)

drug_targets_ppi["HER2_max_pr"].describe()


count    522.000000
mean       0.000191
std        0.000205
min        0.000004
25%        0.000078
50%        0.000134
75%        0.000216
max        0.001975
Name: HER2_max_pr, dtype: float64

In [118]:
# load Luminal A PageRank scores
pr_luma_df = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_PPI_Pagerank.csv",
    index_col=0
)

pr_luma_df.shape, pr_luma_df.head()


((9875, 1),
        pagerank
 TP53   0.002233
 SRC    0.002190
 EP300  0.001869
 EGFR   0.001740
 HRAS   0.001672)

In [120]:
# build gene → pagerank dict for Luminal A
luma_pr = pr_luma_df["pagerank"].to_dict()

# mean pagerank of targets per drug (LumA)
drug_targets_ppi["LumA_mean_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: sum(luma_pr[g] for g in genes) / len(genes)
)

# max pagerank of targets per drug (LumA)
drug_targets_ppi["LumA_max_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: max(luma_pr[g] for g in genes)
)

drug_targets_ppi[["LumA_mean_pr", "LumA_max_pr"]].describe()


,LumA_mean_pr,LumA_max_pr
count,522.000000,522.000000
mean,0.000205,0.000205
std,0.000245,0.000245
min,0.000003,0.000003
25%,0.000067,0.000067
50%,0.000132,0.000132
75%,0.000253,0.000253
max,0.002190,0.002190


In [122]:
# load Luminal B PageRank scores
pr_lumb_df = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_PPI_Pagerank.csv",
    index_col=0
)

pr_lumb_df.shape, pr_lumb_df.head()


((9875, 1),
         pagerank
 SRC     0.001973
 TP53    0.001848
 EGFR    0.001528
 EP300   0.001395
 RPS27A  0.001390)

In [124]:
# build gene → pagerank dict for Luminal B
lumb_pr = pr_lumb_df["pagerank"].to_dict()

# mean pagerank of targets per drug (LumB)
drug_targets_ppi["LumB_mean_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: sum(lumb_pr[g] for g in genes) / len(genes)
)

# max pagerank of targets per drug (LumB)
drug_targets_ppi["LumB_max_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: max(lumb_pr[g] for g in genes)
)

drug_targets_ppi[["LumB_mean_pr", "LumB_max_pr"]].describe()


,LumB_mean_pr,LumB_max_pr
count,522.000000,522.000000
mean,0.000190,0.000190
std,0.000209,0.000209
min,0.000005,0.000005
25%,0.000075,0.000075
50%,0.000131,0.000131
75%,0.000215,0.000215
max,0.001973,0.001973


In [126]:
# load TNBC (Basal) PageRank scores
pr_tnbc_df = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_PPI_Pagerank.csv",
    index_col=0
)

pr_tnbc_df.shape, pr_tnbc_df.head()


((9875, 1),
         pagerank
 SRC     0.001931
 TP53    0.001838
 EGFR    0.001459
 RPS27A  0.001389
 EP300   0.001337)

In [128]:
# build gene → pagerank dict for TNBC
tnbc_pr = pr_tnbc_df["pagerank"].to_dict()

# mean PageRank of targets (TNBC)
drug_targets_ppi["TNBC_mean_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: sum(tnbc_pr[g] for g in genes) / len(genes)
)

# max PageRank of targets (TNBC)
drug_targets_ppi["TNBC_max_pr"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: max(tnbc_pr[g] for g in genes)
)

drug_targets_ppi[["TNBC_mean_pr", "TNBC_max_pr"]].describe()


,TNBC_mean_pr,TNBC_max_pr
count,522.000000,522.000000
mean,0.000189,0.000189
std,0.000204,0.000204
min,0.000004,0.000004
25%,0.000071,0.000071
50%,0.000132,0.000132
75%,0.000218,0.000218
max,0.001931,0.001931


# Target Essentiality (DepMap)

In [131]:
import pandas as pd

depmap = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\CRISPR_gene_effect.csv",
    index_col=0
)

depmap.shape


(1086, 17386)

In [133]:
# clean column names: "TP53 (7157)" → "TP53"
depmap_genes = depmap.copy()
depmap_genes.columns = depmap_genes.columns.str.split(" ").str[0]

# gene-level mean essentiality
gene_essentiality = depmap_genes.mean(axis=0)

gene_essentiality.describe()


count    17386.000000
mean        -0.153986
std          0.378210
min         -2.701632
25%         -0.121551
50%         -0.033647
75%          0.018692
max          0.295341
dtype: float64

In [135]:
# map gene → essentiality
ess_dict = gene_essentiality.to_dict()

# mean essentiality of drug targets
drug_targets_ppi["mean_essentiality"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: sum(ess_dict.get(g, 0) for g in genes) / len(genes)
)

# strongest essential target
drug_targets_ppi["min_essentiality"] = drug_targets_ppi["targets_ppi"].apply(
    lambda genes: min(ess_dict.get(g, 0) for g in genes)
)

drug_targets_ppi[["mean_essentiality", "min_essentiality"]].describe()


,mean_essentiality,min_essentiality
count,522.000000,522.000000
mean,-0.077755,-0.077755
std,0.255713,0.255713
min,-2.286522,-2.286522
25%,-0.080998,-0.080998
50%,-0.012706,-0.012706
75%,0.024017,0.024017
max,0.140201,0.140201
